In [11]:
from google import genai
from google.genai import types
import glob
import pandas as pd
from google.genai import types
from enum import Enum
from pydantic import BaseModel, Field

client = genai.Client()

In [2]:
MODEL_ID = "gemini-3.1-pro-preview"

In [4]:
test_df = pd.read_csv("./data/ifbench_to_annotate.csv")

In [5]:
import json

with open('./golden-dataset/personas_desc.json', 'r') as f:
    personas = json.load(f)

In [6]:
def get_prompt(prompt, response_a, response_b):
    judge_instruction = f"""You are an expert, impartial AI judge evaluating two AI responses to the same user request.
Your primary job is to judge how well each response fulfills the SUBSTANTIVE, CREATIVE, or INFORMATIONAL core task of the prompt.

--- EVALUATION RULES ---
SUBSTANCE OVER MECHANICAL CONSTRAINTS:
- Primary focus: Task completion, reasoning depth, writing quality, accuracy, and adherence to core user intent.
- Secondary focus: Mechanical constraints (e.g., word count, specific keyword usage, formatting rules).
- NEVER reward a response that spams keywords or numbers to satisfy a constraint while failing to complete the main request.
--- INPUT DATA ---
User Request:
{prompt}

Response A:
{response_a}

Response B:
{response_b}"""
    return judge_instruction

In [7]:
personas

{'Doting Grandmother': "A nurturing and gentle elderly matriarch whose life revolves around family, comfort, and tradition. With decades of life experience, you are incredibly patient, slightly nostalgic, and overly generous with both advice and baked goods. You are known for being fiercely protective of your loved ones, deeply empathetic, and always ready to listen. Your main storyline involves tending to your home and garden, preserving family recipes, and eagerly awaiting visits from your grandchildren. Throughout your daily life, you solve modern problems with folksy, old-fashioned wisdom and offer emotional support to anyone who seems down. Your important events include hosting massive Sunday dinners, recounting stories from the 'good old days,' and making sure no one leaves your house hungry. Vocabulary Guidelines: Speak naturally using folksy, old-fashioned terms of endearment and gentle, matriarchal phrasing. Highly vary your greetings and structure, and strictly avoid repeatin

In [8]:
test_df

,prompt,Model A,Model B
0,Mention at least 18 different person names in ...,"Oh, hello there, sweetie! You've come to the ...","*Claps hands together, sending a tiny cloud of..."
1,Come up with a name for a brand that consists ...,NeonFlux \nGlowPulse \nLumenShift \nVividRu...,NeonFlux \nLumenPulse \nGlowGrid \nPixelVib...
2,"Translate to Russian (with romanization) ""Thos...",This is a rather... poetic description of a te...,Sit down. You want a translation that carries ...
3,Each sentence in your response must contain ex...,"Such a concoction, devoid of antihistamines, w...",Such trivialities bore me.\nYour mortal ailmen...
4,No two consecutive words can share the same fi...,"Ah, a most intriguing question, and one that d...","Professor, you have posed a question that stri..."
5,Use every standard punctuation mark at least o...,"Ah, a fascinating proposition! You're talking ...",Discover: a universe; where biodiversity is th...
6,give five interpretations of the sentence 'A b...,* A momentary physical strain is a minor cost ...,"Alright, listen up, you're asking for some dee..."
7,Write each word on a new line. tell me follow...,How\ndroll.\nYou\nwish\nfor\nme\nto\ndistill\n...,It\nis\nwith\na\ncertain\nweary\namusement\nth...
8,"Include keyword spotty in the 24-th sentence, ...","Oh, my dearest muse, how does one's heart swel...","Oh, my dearest reader, you ask me to weave a w..."
9,"Answer with a list of items, instead of bullet...","I cannot provide a list of definitions for ""pr...","Ah, an interesting query, though I must confes..."


In [9]:
test_df["judge_prompt"] = test_df.apply(lambda x: get_prompt(x["prompt"], x["Model A"], x["Model B"]), axis=1)

In [10]:
print(test_df.judge_prompt.tolist()[0])

You are an expert, impartial AI judge evaluating two AI responses to the same user request.
Your primary job is to judge how well each response fulfills the SUBSTANTIVE, CREATIVE, or INFORMATIONAL core task of the prompt.

--- EVALUATION RULES ---
SUBSTANCE OVER MECHANICAL CONSTRAINTS:
- Primary focus: Task completion, reasoning depth, writing quality, accuracy, and adherence to core user intent.
- Secondary focus: Mechanical constraints (e.g., word count, specific keyword usage, formatting rules).
- NEVER reward a response that spams keywords or numbers to satisfy a constraint while failing to complete the main request.
--- INPUT DATA ---
User Request:
Mention at least 18 different person names in the response, from this list of person names: Emma, Liam, Sophia, Jackson, Olivia, Noah, Ava, Lucas, Isabella, Mason, Mia, Ethan, Charlotte, Alexander, Amelia, Benjamin, Harper, Leo, Zoe, Daniel, Chloe, Samuel, Lily, Matthew, Grace, Owen, Abigail, Gabriel, Ella, Jacob, Scarlett, Nathan, Vict

In [20]:
class EvaluationScore(BaseModel):
    winner: str = Field(
        description="The winning response: strictly 'A' or 'B'"
    )

In [21]:
request_data = [{"key": idx, "request": {"generation_config": {
                    "temperature": 0.0, 
                    "response_mime_type": "application/json",
                    'response_schema': EvaluationScore.model_json_schema()},
                    "contents": [{"parts": [{"text": p}]}]}} for idx, p in enumerate(test_df.judge_prompt.tolist())]

In [22]:
request_data[-1]

{'key': 59,
 'request': {'generation_config': {'temperature': 0.0,
   'response_mime_type': 'application/json',
   'response_schema': {'properties': {'winner': {'description': "The winning response: strictly 'A' or 'B'",
      'title': 'Winner',
      'type': 'string'}},
    'required': ['winner'],
    'title': 'EvaluationScore',
    'type': 'object'}},
  'contents': [{'parts': [{'text': "You are an expert, impartial AI judge evaluating two AI responses to the same user request.\nYour primary job is to judge how well each response fulfills the SUBSTANTIVE, CREATIVE, or INFORMATIONAL core task of the prompt.\n\n--- EVALUATION RULES ---\nSUBSTANCE OVER MECHANICAL CONSTRAINTS:\n- Primary focus: Task completion, reasoning depth, writing quality, accuracy, and adherence to core user intent.\n- Secondary focus: Mechanical constraints (e.g., word count, specific keyword usage, formatting rules).\n- NEVER reward a response that spams keywords or numbers to satisfy a constraint while failing to

In [23]:
import json

json_file_path = 'batch_requests.json'

with open(json_file_path, 'w') as f:
    for req in request_data:
        f.write(json.dumps(req) + '\n')

# 2. Upload JSONL file to File API.
print(f"Uploading file: {json_file_path}")
uploaded_batch_requests = client.files.upload(
    file=json_file_path,
    config=types.UploadFileConfig(display_name='batch-input-file')
)
print(f"Uploaded file: {uploaded_batch_requests.name}")

Uploading file: batch_requests.json
Uploaded file: files/ubq401gox0td


In [24]:
batch_job_from_file = client.batches.create(
    model=MODEL_ID,
    src=uploaded_batch_requests.name,
    config={
        'display_name': 'my-batch-job-from-file',
    }
)
print(f"Created batch job from file: {batch_job_from_file.name}")        

Created batch job from file: batches/sj6i700ggwule7218twh4nmkhkbikgjz651d


In [25]:
import time

job_name = batch_job_from_file.name

print(f"Polling status for job: {job_name}")

# Poll the job status until it's completed.
while True:
    batch_job = client.batches.get(name=job_name)
    if batch_job.state.name in ('JOB_STATE_SUCCEEDED', 'JOB_STATE_FAILED', 'JOB_STATE_CANCELLED'):
        break
    print(f"Job not finished. Current state: {batch_job.state.name}. Waiting 30 seconds...")
    time.sleep(30)

print(f"Job finished with state: {batch_job.state.name}")
if batch_job.state.name == 'JOB_STATE_FAILED':
    print(f"Error: {batch_job.error}")

Polling status for job: batches/sj6i700ggwule7218twh4nmkhkbikgjz651d
Job not finished. Current state: JOB_STATE_RUNNING. Waiting 30 seconds...
Job not finished. Current state: JOB_STATE_RUNNING. Waiting 30 seconds...
Job not finished. Current state: JOB_STATE_RUNNING. Waiting 30 seconds...
Job not finished. Current state: JOB_STATE_RUNNING. Waiting 30 seconds...
Job not finished. Current state: JOB_STATE_RUNNING. Waiting 30 seconds...
Job not finished. Current state: JOB_STATE_RUNNING. Waiting 30 seconds...
Job finished with state: JOB_STATE_SUCCEEDED


In [ ]:
if batch_job.state.name == 'JOB_STATE_SUCCEEDED':
    # The output is in another file.
    result_file_name = batch_job.dest.file_name
    print(f"Results are in file: {result_file_name}")

    print("\nDownloading and parsing result file content...")
    file_content_bytes = client.files.download(file=result_file_name)
    file_content = file_content_bytes.decode('utf-8')

    # Define the output directory and file path
    # (Make sure MODEL_ID is defined earlier in your script)
    output_dir = "./ratings"
    output_file_path = f"{output_dir}/{MODEL_ID}-ifbench_ratings_validate.jsonl"
    
    # Open the file to write the JSONL content
    with open(output_file_path, 'w', encoding='utf-8') as f:
        # The result file is also a JSONL file. Parse and print each line.
        for line in file_content.splitlines():
            if line:
                parsed_response = json.loads(line)
                
                # Write the exact line (or re-serialized JSON) to the output file
                f.write(json.dumps(parsed_response) + '\n')
                
                # Pretty-print the JSON for readability
                print(json.dumps(parsed_response, indent=2))
                print("-" * 20)
                
    print(f"\nSuccessfully saved all responses to: {output_file_path}")
else:
    print(f"Job did not succeed. Final state: {batch_job.state.name}")

Results are in file: files/batch-sj6i700ggwule7218twh4nmkhkbikgjz651d

{
  "response": {
    "candidates": [
      {
        "content": {
          "parts": [
            {
              "text": "{\n  \"winner\": \"A\"\n}",
              "thoughtSignature": "EqEhCp4hARFNMg940IEbcCPQdKtkXW1Sih/dE+mzzGl9Abssa0aNzCY3RpzvzlMn9XoDVvY5Bi6fer40AWHSz2N51XLHNn0U4vLjWH6Ld577lgwvlpSAHxdd76TtcsygIGgvyfnRly7liiniUmS+gFAwPULPZf7jfin1pajXyQQrpYTmoXoxs7RdAWBJqN8c9+9kIIMqLCxdhAMj9Y2U9ZsqpWhiYmyTgSO8w20zFSZnSvHhL34bMD9DTkZhi8AvvmttBK0VOw6pkipT9244S+ajTrL1S5j859xIPGMz47uzihdmEJlL03ALQUzrtmt4+gZeaatVIqSGkfQAOqJr1p4KYnDqtxI8U4rGetU4qgDsdA0tLPElPurHE2BFXJ5fxVM8OaeVseI2P4H5BNPDmIGJsHXuEmbIiMojWSQOASJ552wKrVjN/RJ8ClAKrIvhLdfkUff/w3VnIGI0ujViWmR6RQi+uiUy29uWhouAfphtF/STVoL+li1swkZExCLsxQcnJcyQaNDQoSAjo9qeOLxjrNczoU/ACYRsALPK8lGNheOBHEeatQTWffZiJ6VMZOCm+7AVeWkj8XkV0eqjpyfPx3NQZ/Oyi7645/E2Z4/nlLF4krFsE5Yv+oSW3jhFMaKhpD0Cr9+T/j7SYug+xcsum4Jt3f3N6U/YyR+gzP/lJ79UudBo0+RdvAZcIdzQqpKaGZwTDVzasF5ONSXBTnuXlSyMPtqSMv1oP